In [ ]:
import os
from pathlib import Path
if Path.cwd().name.startswith("step"):
    os.chdir("..")


# Step 1 — Get the data from DBLP

In this step, our goal is to gather information about researchers from the LRE (EC).

We have access to a provided file `2026_05_LRE_EC.csv` that contains several information about ECs. We will use it to get more data.

In [2]:
import pandas as pd
import re

import requests
import time
import pathlib

Let's start by looking at the data we have

In [20]:
lre = pd.read_csv("data/2026_05_LRE_EC.csv", sep=";")

lre.head()

,Clef,Prénom,Nom,PrénomNom,Présent aujourd'hui,Doctorat,Statut,Catégorie,Entité,Équipe,Date de début,Date de fin,Date de début en année,Lien DBLP,Lien Google Scholar
0,AA,Amazigh,Amrane,Amazigh Amrane,1,Dr,MCF,Perm,LRE,AA,01/09/2022,01/01/3000,2022,https://dblp.org/pid/237/1676.html,https://scholar.google.com/citations?user=V9xJ...
1,AB,Aymeric,Brody,Aymeric Brody,1,Dr,MCF,Perm,LRE,MNSHS,17/10/2022,01/01/3000,2022,pas de page DBLP,https://scholar.google.com/citations?hl=fr&use...
2,AD,Akim,Demaille,Akim Demaille,0,Dr,MCF,Perm,LRDE,AA,01/05/1999,08/02/2017,1999,https://dblp.org/pid/14/133.html,NaN
3,ADL,Alexandre,Duret-Lutz,Alexandre Duret-Lutz,1,Dr,HDR,Perm,LRDE-LRE,AA,10/02/2017,01/01/3000,2007,https://dblp.org/pid/43/6032.html,https://scholar.google.com/citations?hl=fr&use...
4,AP,Adrien,Pommellet,Adrien Pommellet,1,Dr,MCF,Perm,LRDE-LRE,AA,04/03/2019,01/01/3000,2019,https://dblp.org/pid/125/0224.html,https://scholar.google.com/citations?hl=fr&use...


We notice that some ECs have missing DBLP pages and/or Google Scholar pages.

Also, some ECs are marked as not present in the LRE anymore, we will filter them out.

In [21]:
lre = lre[lre["Présent aujourd'hui"] == 1]
lre.head()

,Clef,Prénom,Nom,PrénomNom,Présent aujourd'hui,Doctorat,Statut,Catégorie,Entité,Équipe,Date de début,Date de fin,Date de début en année,Lien DBLP,Lien Google Scholar
0,AA,Amazigh,Amrane,Amazigh Amrane,1,Dr,MCF,Perm,LRE,AA,01/09/2022,01/01/3000,2022,https://dblp.org/pid/237/1676.html,https://scholar.google.com/citations?user=V9xJ...
1,AB,Aymeric,Brody,Aymeric Brody,1,Dr,MCF,Perm,LRE,MNSHS,17/10/2022,01/01/3000,2022,pas de page DBLP,https://scholar.google.com/citations?hl=fr&use...
3,ADL,Alexandre,Duret-Lutz,Alexandre Duret-Lutz,1,Dr,HDR,Perm,LRDE-LRE,AA,10/02/2017,01/01/3000,2007,https://dblp.org/pid/43/6032.html,https://scholar.google.com/citations?hl=fr&use...
4,AP,Adrien,Pommellet,Adrien Pommellet,1,Dr,MCF,Perm,LRDE-LRE,AA,04/03/2019,01/01/3000,2019,https://dblp.org/pid/125/0224.html,https://scholar.google.com/citations?hl=fr&use...
5,AQK,Abdul Qadir,Khan,Abdul Qadir Khan,1,Dr,MCF,Perm,LRE,SÉCUSYS,01/04/2026,01/01/3000,2026,https://dblp.org/pid/373/0583.html,https://scholar.google.fr/citations?hl=fr&user...


Now we will isolate each DBLP PID.

In [22]:
def extract_pid_list(string):
    string_list = string.split(" ")
    result = []
    for s in string_list:
        match = re.findall(r'.*pid\/(.*).html', s)
        if match is not None:
            result.extend(match)
    return result if result else None

In [23]:
lre["DBLP_PID"] = [extract_pid_list(dblp_link) for dblp_link in lre["Lien DBLP"]]
lre["DBLP_PID"].head()

0    [237/1676]
1          None
3     [43/6032]
4    [125/0224]
5    [373/0583]
Name: DBLP_PID, dtype: object

In [ ]:
def build_ec_csv():
    ec_csv = pd.DataFrame()

    ec_csv["full_name"] = lre["PrénomNom"]
    ec_csv["dblp_pids"] = lre["DBLP_PID"].apply(lambda pids: [pid.replace("/", "_") for pid in pids] if pids is not None else None)
    ec_csv["dblp_pages"] = lre["Lien DBLP"].apply(lambda link: link.split(" ; ") if link != "pas de page DBLP" else None)
    ec_csv["team"] = lre["Équipe"]
    ec_csv["arrival_year"] = lre["Date de début en année"]
    return ec_csv

ec_csv = build_ec_csv()
ec_csv.head()

,full_name,dblp_pids,dblp_pages,team,arrival_year
0,Amazigh Amrane,[237/1676],[https://dblp.org/pid/237/1676.html ],AA,2022
1,Aymeric Brody,None,None,MNSHS,2022
3,Alexandre Duret-Lutz,[43/6032],[https://dblp.org/pid/43/6032.html],AA,2007
4,Adrien Pommellet,[125/0224],[https://dblp.org/pid/125/0224.html],AA,2019
5,Abdul Qadir Khan,[373/0583],[https://dblp.org/pid/373/0583.html],SÉCUSYS,2026


In [25]:
ec_csv.to_csv("data/ec.csv")

We can see that we have some NaN values for the PIDs, meaning we have non existing DBLP pages for some ECs. This will be taken into account in the knowledge graph.

We will now need to download and cache information about the ECs with the previously extracted PIDs.

To do so, we will need to access the ECs' record, available at `https://dblp.org/pid/{EC_PID}.xml` and cache it in the `raw/` folder.

In [ ]:
RAW_DIR = pathlib.Path("data/raw")
RAW_DIR.mkdir(exist_ok=True)

def download_dblp_record(author):
    """
    Downloads the DBLP XML record for a given PID and caches it.
    Example pid format expected: '237/1676'
    """
    pid_list = author["DBLP_PID"]
    if pid_list is not None:
        for pid in pid_list:
            if not isinstance(pid, str) :
                return None
            safe_pid = pid.replace('/', '_')
            file_path = RAW_DIR / f"author_{safe_pid}.xml"
            
            if file_path.exists():
                print(f"Loaded from cache: {pid}")
                return file_path.read_text(encoding='utf-8')
            
            print(f"Downloading: {pid}...")
            url = f"https://dblp.org/pid/{pid}.xml"
            
            headers = {
                "User-Agent": "LRE-KG-course/1.0"
            }
            
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            file_path.write_text(response.text, encoding='utf-8')
            
            time.sleep(2.5)

Now that we have this function that downloads the xml record for one pid, we will execute it for all ECs.

In [27]:
count = 0
for idx, row in lre.iterrows():
    print(row)
    if download_dblp_record(row) is not None:
        count += 1

print(f"Loaded/Downloaded {count} records out of {len(lre)} ECs.")

Clef                                                                     AA
Prénom                                                              Amazigh
Nom                                                                  Amrane
PrénomNom                                                    Amazigh Amrane
Présent aujourd'hui                                                       1
Doctorat                                                                 Dr
Statut                                                                  MCF
Catégorie                                                              Perm
Entité                                                                  LRE
Équipe                                                                   AA
Date de début                                                    01/09/2022
Date de fin                                                      01/01/3000
Date de début en année                                                 2022
Lien DBLP   

We have downloaded the data of 46 ECs out of 47, there is one missing because the person doesn't have a DBLP page.

The DBLP xml data is available in the `data/raw/` folder at the root.